# Group-wise comparisons

Compare the relative abundance and protein signatures of pancreatic cell types across donors and disease groups. Each section contributes one donor observation.

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/04_group_comparisons.ipynb).

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
from phenotype_signatures import (hierarchy_tables, phenotype_colors, phenotype_key, names_in_order, subtype_display_name, MACROPHAGE, full_panel)
release = validate_release(data)

DONOR = "115"                      # Any donor listed in the manifest
DISEASE = "ND"                     # ND, AAb+, or T1D
LEVEL = "subtype"                   # Marker profiles: subtype, primary, detail, immune_class, t_subtype, macrophage_profile
UMAP_MARKERS = ["GCG", "INS", "SST", "Podoplanin", "B3TUBB", "CD56", "CD31", "Ker8_18", "CD3e"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
inks = phenotype_colors(counts, "Pancreas")
counts, evidence = hierarchy_tables(counts, evidence)

## 1. Cell-subtype composition by donor

Each bar partitions all admitted cells by their most specific recorded assignment. The disease-group panels show one point per comparable donor and a horizontal line at the median. The table retains counts from assessable sections and marks unavailable calls as missing.

In [ ]:
subtypes = counts.loc[counts.level.eq("subtype")].copy()
wide = subtypes.pivot(index=["disease","donor"], columns="label", values="cells")
admitted = subtypes.groupby(["disease","donor"], observed=True).eligible_cells.first().reindex(wide.index)
assert wide.sum(axis=1).eq(admitted).all()
order = names_in_order(wide.columns, "Pancreas")
donor_order = sorted(wide.index, key=lambda x: (group_order.index(x[0]), int(x[1])))
wide = wide.loc[donor_order, order]
frac = wide.div(admitted, axis=0).reindex(wide.index)
comparable = subtypes.pivot(index=["disease","donor"], columns="label", values="comparable")
group_frac = frac.where(comparable)
legend_height = .23 * np.ceil(len(order)/3) + .3
fig = plt.figure(figsize=(13, 5 + legend_height), layout="constrained")
grid = fig.add_gridspec(2, 1, height_ratios=[4,legend_height])
ax = fig.add_subplot(grid[0])
bottom = np.zeros(len(frac))
for label in order:
    heights = 100 * frac[label].fillna(0).to_numpy()
    ax.bar(np.arange(len(frac)), heights, bottom=bottom, width=.83, color=inks[label])
    bottom += heights
ax.set(xticks=range(len(frac)), xticklabels=frac.index.get_level_values("donor"), ylim=(0,100),
       ylabel="Admitted cells (%)", xlabel="Pancreas donor", xlim=(-.7,len(frac)-.3))
ax.tick_params(axis="x", rotation=60)
for disease in group_order:
    loc = np.flatnonzero(frac.index.get_level_values("disease") == disease)
    ax.text(loc.mean(),103,disease,ha="center",color=group_ink[disease],weight="bold",size=9)
phenotype_key(fig.add_subplot(grid[1]), order, inks, columns=3)
plt.show()
display(pd.DataFrame({"donor cells":wide.xs(DONOR,level="donor").iloc[0],
                      "donor %":100*frac.xs(DONOR,level="donor").iloc[0],
                      f"{DISEASE} mean donor %":100*group_frac.xs(DISEASE).mean(),
                      f"{DISEASE} comparable donors":group_frac.xs(DISEASE).count()}).round(3))

In [ ]:
# Each panel has a subtype-specific y-axis so rare cells remain visible.
for start in range(0, len(order), 12):
    panel = order[start:start+12]
    nrows = int(np.ceil(len(panel)/3))
    fig, axes = plt.subplots(nrows,3,figsize=(13,2.7*nrows),squeeze=False,layout="constrained")
    for ax,label in zip(axes.flat,panel):
        for i,g in enumerate(group_order):
            v = 100 * group_frac.xs(g, level="disease")[label].dropna().sort_index()
            if v.empty:
                continue
            ax.scatter(i + np.linspace(-.15,.15,len(v)), v, s=21, color=group_ink[g],alpha=.8)
            ax.hlines(v.median(), i-.24, i+.24, color="#20282b", linewidth=1.6)
        ax.set(xticks=range(3),xticklabels=group_order,title=subtype_display_name(label),
               xlim=(-.45,2.45),ylim=(0,None),ylabel="Admitted cells (%)")
    for ax in list(axes.flat)[len(panel):]:
        ax.set_axis_off()
    plt.show()

## 2. Full-panel subtype expression by disease group

Each disease group has a dotplot and heatmap spanning the complete subtype census and calibrated panel. Dot color scales each marker across the displayed rows within that group. Dot area gives the measured fraction ≥0.10. For absolute comparisons between disease groups use the heatmap or calibrated means in `group_measurements`. Row labels give cell and donor support. The default weights donors equally. Set `WEIGHTING = "cells"` to pool cells.

In [ ]:
WEIGHTING = "donors"
group_measurements = {}
for group in group_order:
    group_measurements[group] = show_profiles(counts, evidence, level=LEVEL, disease=group,
                                            weighting=WEIGHTING)
display(group_measurements[DISEASE].reset_index())

## 3. Between-donor protein differences

Each row shows one donor and its assigned phenotype count. A donor with one or two cells is shown explicitly. A measured zero assignment has a 0-cell row and cannot provide a within-phenotype expression mean. Gray marks unmeasured or noncomparable combinations. Full marker-specific counts and positive fractions follow the plots.

In [ ]:
PHENOTYPE = "Beta" if LEVEL in ("subtype","primary","detail") else {
    "immune_class":"Imm-T-cell", "t_subtype":"T-CD8", "macrophage_profile":"Macrophage_Gal3+"
}[LEVEL]
available_labels = counts.loc[counts.level.eq(LEVEL),"label"].unique()
if PHENOTYPE not in available_labels:
    raise ValueError(f"Select a phenotype from {sorted(available_labels)}")
markers = MACROPHAGE if LEVEL == "macrophage_profile" else full_panel(evidence, tissue="Pancreas")
selected = donor_heatmap(counts, evidence, tissue="Pancreas", level=LEVEL, label=PHENOTYPE, markers=markers)
display(selected.loc[selected.index.get_level_values("marker").isin(markers)])

### Remaining endocrine conflicts by disease group

Compare like hormone pairs and native decision reasons across sections.

In [ ]:
for group in group_order:
    print(group)
    display(conflict_summary(data, counts, disease=group))